In [ ]:
import os
import duckdb
from pathlib import Path
from dotenv import load_dotenv

In [ ]:
notebook_dir = Path.cwd()
repo_root = notebook_dir.parent
load_dotenv(repo_root / ".env")

con = duckdb.connect(str(repo_root / os.getenv("DUCKDB_PATH", "data/f1_local.duckdb")))

In [ ]:
con.sql("CREATE SCHEMA IF NOT EXISTS bronze")
con.sql("CREATE SCHEMA IF NOT EXISTS silver")
con.sql("CREATE SCHEMA IF NOT EXISTS gold")

In [ ]:
schemas = con.sql("SELECT schema_name FROM information_schema.schemata").fetchall()
print(schemas)
con.close()

In [ ]:
repo_root = Path.cwd().parent
load_dotenv(repo_root / ".env")

con = duckdb.connect(str(repo_root / os.getenv("DUCKDB_PATH")))

In [ ]:
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
    SET s3_access_key_id = '{os.getenv("AWS_ACCESS_KEY_ID")}';
    SET s3_secret_access_key = '{os.getenv("AWS_SECRET_ACCESS_KEY")}';
    SET s3_region = '{os.getenv("AWS_REGION")}';
""")
bucket = os.getenv("S3_BUCKET_RAW")
print("DuckDB connected to S3 success")

In [ ]:
checks = {
    "race_results": {"expected_rounds": 2, "key_col": "position"},
    "driver_standings": {"expected_rounds": 1, "key_col": "points"},
    "constructor_standings": {"expected_rounds": 1, "key_col": "points"},
    "race_schedule": {"expected_rounds": 22, "key_col": "race_date"},
}

for endpoint, dict in checks.items():
    path = f"s3://{bucket}/jolpica/2026/{endpoint}.parquet"

    row_count = con.execute(
        f"SELECT COUNT(*) FROM read_parquet('{path}')"
    ).fetchone()[0]

    round_count = con.execute(
        f"SELECT COUNT(DISTINCT round) FROM read_parquet('{path}')"
    ).fetchone()[0]

    null_count = con.execute(
        f"SELECT COUNT(*) FROM read_parquet('{path}') "
        f"WHERE {dict['key_col']} IS NULL"
    ).fetchone()[0]

    status = "✓" if round_count == dict["expected_rounds"] and null_count == 0 else "✗"
    print(
        f"{status} {endpoint:<25} "
        f"rows={row_count:<6} "
        f"rounds={round_count}/22   "
        f"nulls_on_{dict['key_col']}={null_count}"
    )

In [ ]:
path = f"s3://{bucket}/jolpica/2026/driver_standings.parquet"

result = con.execute(f"""
    SELECT driver_name, points, wins
    FROM read_parquet('{path}')
    WHERE round = 2
    ORDER BY position
    LIMIT 3
""").df()
print(result.to_string(index=False))

In [ ]:
import sys, os, duckdb
from pathlib import Path
from dotenv import load_dotenv
repo_root = Path.cwd().parent
sys.path.insert(0, str(repo_root))
load_dotenv(repo_root / ".env")
from ingestion.jolpica.bronze_loader import load_jolpica_bronze, verify_bronze_tables

In [ ]:
bucket = os.getenv("S3_BUCKET_RAW")

In [ ]:
con = duckdb.connect(str(repo_root / os.getenv("DUCKDB_PATH")))
load_jolpica_bronze(con, year=2026)
verify_bronze_tables(con)

In [5]:
import sys, os, duckdb
from pathlib import Path
from dotenv import load_dotenv
repo_root = Path.cwd().parent.parent
sys.path.insert(0, str(repo_root))
load_dotenv(repo_root / ".env")

True

In [6]:
con = duckdb.connect(str(repo_root / os.getenv("DUCKDB_PATH")))

In [ ]:
con.sql("SELECT * FROM silver.fastf1_telemetry LIMIT 5")

In [ ]:
con.sql("SELECT * FROM silver.fastf1_laps LIMIT 5")

In [7]:
# Failing pit stops
con.sql("""
    select
        pit_stop_id,
        season,
        round,
        driver_id,
        pit_stop_number,
        pit_stop_duration_seconds
    from gold.mart_pit_stops
    where pit_stop_duration_seconds < 1.5
       or pit_stop_duration_seconds > 600
    order by pit_stop_duration_seconds
""")

┌──────────────────────────────────┬────────┬───────┬───────────┬─────────────────┬───────────────────────────┐
│           pit_stop_id            │ season │ round │ driver_id │ pit_stop_number │ pit_stop_duration_seconds │
│             varchar              │ int64  │ int64 │  varchar  │      int64      │          double           │
├──────────────────────────────────┼────────┼───────┼───────────┼─────────────────┼───────────────────────────┤
│ ea6c78a51d9d5c96e91e0c1911bfed47 │   2026 │     1 │ alonso    │               2 │                   972.356 │
│ 7414819c9f35fdf5823134f4fd3f8a06 │   2026 │     1 │ stroll    │               3 │                  1081.553 │
└──────────────────────────────────┴────────┴───────┴───────────┴─────────────────┴───────────────────────────┘

In [8]:
# Failing degradation rates
con.sql("""
    select
        season,
        round,
        driver,
        stint,
        compound,
        stint_length_laps,
        degradation_rate_seconds_per_lap
    from gold.mart_tyre_strategy
    where degradation_rate_seconds_per_lap is not null
      and (
            degradation_rate_seconds_per_lap < -2.0
         or degradation_rate_seconds_per_lap > 5.0
      )
    order by degradation_rate_seconds_per_lap
""")

┌────────┬───────┬─────────┬───────┬──────────┬───────────────────┬──────────────────────────────────┐
│ season │ round │ driver  │ stint │ compound │ stint_length_laps │ degradation_rate_seconds_per_lap │
│ int64  │ int64 │ varchar │ int64 │ varchar  │       int64       │              double              │
├────────┼───────┼─────────┼───────┼──────────┼───────────────────┼──────────────────────────────────┤
│   2026 │     1 │ bot     │     2 │ HARD     │                 3 │               -6.390500000000003 │
│   2026 │     2 │ oco     │     3 │ SOFT     │                 9 │               -2.981316666666667 │
│   2026 │     1 │ str     │     5 │ SOFT     │                 4 │              -2.0895999999999986 │
│   2026 │     1 │ alo     │     2 │ SOFT     │                 1 │                              nan │
│   2026 │     2 │ had     │     1 │ SOFT     │                 1 │                              nan │
│   2026 │     2 │ alo     │     2 │ MEDIUM   │                 1 │      

In [9]:
con.close()